# Fix a mosaic-shift-induced FOV gap

A worked example of diagnosing and patching a real acquisition-planning bug,
kept as a template for the same class of problem on other samples --
`notebooks/tests/` holds notebooks like this one, built to investigate/fix a
specific real issue rather than run as a standing part of the numbered
`prepare_imaging` pipeline.

**The bug this fixes**: the usual workflow is (1) scan a low-mag (10x) Steve
mosaic of the whole coverslip, (2) image a handful of FOVs at the real
imaging objective ("60x" here) over part of that scan to measure the fixed
stage-calibration offset between the two objectives, (3) shift the 10x
mosaic's stage positions by that offset so it lines up with real (60x)
coordinates, THEN (4) derive the tissue boundary and FOV grid from the
shifted mosaic. If step 3 is skipped (or the shift is computed but applied
too late, after the boundary was already saved), the boundary -- and every
FOV grid built from it -- ends up offset from where the tissue actually is.

**Sections**:
1. Load the raw Steve mosaic tiles (cached locally -- a slow, many-small-file
   read over a network drive) and plot every tile's position, colored by
   objective, to visually confirm objectives 1 and 2 above disagree.
2. Apply the known `(SHIFT_DX_UM, SHIFT_DY_UM)` correction to the low-mag
   tiles only and re-plot -- objectives should now visually line up.
3. Assemble the shifted mosaic and overlay the CURRENT (already-imaging)
   positions file on top of it, to see the real-world misalignment directly.
4. Re-run tissue segmentation + FOV-grid generation on the shifted mosaic,
   using the exact parameters this sample's own local `02_create_boundary_
   from_mosaic.ipynb`/`02_create_positions_from_boundaries.ipynb` were run
   with (copied verbatim below, not re-derived).
5. Overlay OLD vs. NEW FOV positions.
6. Classify every NEW FOV as already-covered (>50% area overlap with its
   nearest OLD FOV) or MISSING, and count/report the missing ones.
7. Append the missing FOVs -- in their own scan order, a contiguous run at
   the boundary -- to the end of the current positions array and save a new
   `positions_{tag}_added.txt`, ready to be imaged in a follow-up loop.

Does not touch the original positions file -- only ever writes a new,
distinctly-named one, and every plot is saved to `SAMPLE_DIR/figures/` for
visual review before trusting the result.

## 1 — Setup

In [ ]:
import os
import sys
import pickle
import dataclasses
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial import cKDTree

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/tests/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import save_positions_array
from MERci.acquisition.configs    import get_fov_geometry
from MERci.acquisition.mosaic     import (
    load_steve_mosaic, assemble_mosaic_canvas, segment_mosaic_tissue, plot_mosaic_segmentation,
)
from MERci.acquisition.positions  import load_hole_polygons, build_boundary_path, get_path_stats

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
POSITIONS_DIR = SAMPLE_DIR / "positions"
FIGURES_DIR   = SAMPLE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_NAME = "fix_mosaic_shift_missing_fovs"
CACHE_DIR     = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Mosaic source ──────────────────────────────────────────────────────────
MOSAIC_DIR  = SAMPLE_DIR / "data" / "mosaic10x"
MOSAIC_NAME = None   # None = auto-detect the only *.msc file present

# ── The known 10x -> real-imaging-objective calibration shift ─────────────
# Externally measured (comparing a handful of real-imaging-objective FOVs
# against the same tissue features in the 10x scan) -- NOT computed by this
# notebook, just applied. Added to every LOW-MAG tile's own (x_um, y_um).
LOW_MAG_OBJECTIVE  = "10x"
HIGH_MAG_OBJECTIVE = "60x"
SHIFT_DX_UM = 410.0
SHIFT_DY_UM = 420.0

# ── Segmentation parameters -- copied verbatim from this sample's own local
# 02_create_boundary_from_mosaic.ipynb (MERci/notebooks/prepare_imaging/
# lineage_tracing/merfish/), NOT re-derived, so the corrected boundary is
# produced the same way the original (uncorrected) one was. ──────────────
MOSAIC_KEEP_OBJECTIVES = [LOW_MAG_OBJECTIVE]   # exclude the high-mag calibration tiles from segmentation
WORKING_PIXEL_UM       = 5.0
THRESHOLD              = 400
SMOOTH_SIGMA_UM        = 10.0
CLOSE_RADIUS_UM        = 50.0
OPEN_RADIUS_UM         = 15.0
MARGIN_UM              = 25.0
MIN_TISSUE_AREA_UM2    = 4_000_000.0
MIN_HOLE_AREA_UM2      = 12_000.0
MIN_ISLAND_AREA_UM2    = 50_000.0
SIMPLIFY_TOL_UM        = 15.0

# ── FOV-grid parameters -- copied verbatim from this sample's own local
# 02_create_positions_from_boundaries.ipynb. ──────────────────────────────
MICROSCOPE           = "ST2"
non_overlap_fraction = 0.9
SCAN_DIRECTION       = "vertical"
pixel_size_um, image_size_px = get_fov_geometry(MICROSCOPE)
step_size_um = pixel_size_um * image_size_px * non_overlap_fraction
fov_size_um  = pixel_size_um * image_size_px

# ── Missing-FOV classification ────────────────────────────────────────────
OVERLAP_THRESHOLD = 0.5   # a NEW FOV counts as "already covered" if its
                           # square tile overlaps its nearest OLD FOV's own
                           # tile (both fov_size_um x fov_size_um) by more
                           # than this fraction of one tile's area.

CURRENT_POSITIONS_PATH = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"
ADDED_POSITIONS_PATH   = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_added.txt"

print(f"Sample name        : {SAMPLE_NAME}")
print(f"step_size_um       : {step_size_um:.1f}")
print(f"fov_size_um        : {fov_size_um:.1f}")
print(f"Shift (~{SHIFT_DX_UM/step_size_um:.2f} x {SHIFT_DY_UM/step_size_um:.2f} FOVs): "
      f"dX={SHIFT_DX_UM}, dY={SHIFT_DY_UM} um")
print(f"Current positions  : {CURRENT_POSITIONS_PATH}")
print(f"Will write         : {ADDED_POSITIONS_PATH}")

## 3 — Load the raw Steve mosaic (cached -- slow over a network drive)

Reading every tile's own `.stv` pickle is many small file opens over
(typically) a network-mounted acquisition drive -- slow regardless of total
data size (dominated by per-file latency, not throughput). Cached to
`analysis/cache/fix_mosaic_shift_missing_fovs/steve_tiles.pkl`, invalidated
by the `.msc` manifest's own mtime, per `NOTEBOOK_GUIDELINES.md` #2/#3.

In [ ]:
msc_candidates = sorted(MOSAIC_DIR.glob(f"{MOSAIC_NAME or '*'}.msc"))
if not msc_candidates:
    raise FileNotFoundError(f"No .msc mosaic manifest found in {MOSAIC_DIR}.")
if len(msc_candidates) > 1:
    print(f"WARNING: {len(msc_candidates)} .msc files found, using the first: {msc_candidates[0].name}.")
MSC_PATH = msc_candidates[0]

tiles_cache = CACHE_DIR / "steve_tiles.pkl"
msc_mtime   = MSC_PATH.stat().st_mtime

cached_ok = False
if tiles_cache.exists():
    with open(tiles_cache, "rb") as fh:
        cached = pickle.load(fh)
    cached_ok = cached.get("msc_mtime") == msc_mtime
    if cached_ok:
        tiles_all = cached["tiles"]
        print(f"Loaded {len(tiles_all)} cached tile(s): {tiles_cache}")

if not cached_ok:
    print(f"Reading {MSC_PATH} (every tile's own .stv file -- can take a couple of minutes "
          f"over a network drive, cached afterward)...")
    tiles_all = load_steve_mosaic(MSC_PATH)
    with open(tiles_cache, "wb") as fh:
        pickle.dump({"msc_mtime": msc_mtime, "tiles": tiles_all}, fh)
    print(f"Loaded and cached {len(tiles_all)} tile(s): {tiles_cache}")

obj_counts = Counter(t.objective_name for t in tiles_all)
print(f"Objective breakdown: {dict(obj_counts)}")
for obj in (LOW_MAG_OBJECTIVE, HIGH_MAG_OBJECTIVE):
    if obj not in obj_counts:
        raise ValueError(f"No tiles found for objective={obj!r} -- check LOW_MAG_OBJECTIVE/"
                          f"HIGH_MAG_OBJECTIVE against the breakdown above.")

## 4 — Step 1: plot every tile's position, low-mag vs. high-mag

First, the coarse picture: every tile's own CENTER position, low-mag vs.
high-mag, over the whole scan -- confirms the high-mag calibration tiles
sit within the low-mag scan's own footprint (not e.g. a mixup between
mosaics) before looking at real image content below. A shift of a few
hundred um is invisible at this whole-mosaic scale (tile spacing is much
larger than the shift), so this step is a sanity check, not the real
alignment test -- that's step 2, which overlays actual pixel content.

In [ ]:
low_tiles  = [t for t in tiles_all if t.objective_name == LOW_MAG_OBJECTIVE]
high_tiles = [t for t in tiles_all if t.objective_name == HIGH_MAG_OBJECTIVE]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter([t.x_um for t in low_tiles],  [t.y_um for t in low_tiles],
           s=8, c="tab:blue", label=f"{LOW_MAG_OBJECTIVE} ({len(low_tiles)})")
ax.scatter([t.x_um for t in high_tiles], [t.y_um for t in high_tiles],
           s=24, c="tab:red", marker="x", label=f"{HIGH_MAG_OBJECTIVE} ({len(high_tiles)})")
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
ax.set_title(f"{SAMPLE_NAME}: raw mosaic tile positions (uncorrected)")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step1_raw_tile_positions.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step1_raw_tile_positions.png'}")

## 5 — Step 2: does the high-mag patch's real footprint sit on tissue?

The actual alignment test, done as a purely SPATIAL check rather than
painting both objectives' pixel values into one image: the two objectives
were shot at very different exposures (the high-mag patch is badly
saturated relative to the low-mag background when composited together,
which makes a shared-intensity overlay hard to read on its own merits).
Instead: assemble the LOW-MAG tiles near the high-mag patch alone (at a
finer `ZOOM_PIXEL_UM` than the whole-mosaic canvas, so real tissue texture
is visible), then draw the high-mag patch's own bounding box as a dashed
outline on top -- a real tissue outline, not painted pixel content. If the
outline sits on continuous gray tissue texture (not the black background
between/around tiles), that position is plausible; comparing BEFORE vs.
AFTER the shift (the outline itself is fixed -- the high-mag tiles are the
calibration reference and never move; the LOW-MAG canvas underneath shifts
instead) shows which one actually lands on real tissue.

In [ ]:
ZOOM_PIXEL_UM = 1.5   # finer than WORKING_PIXEL_UM -- this crop is small, so it's affordable
ZOOM_MARGIN_UM = 400.0   # extra low-mag context around the high-mag patch's own bounding box

high_xs = np.array([t.x_um for t in high_tiles]); high_ys = np.array([t.y_um for t in high_tiles])
patch_half_um = high_tiles[0].image.shape[0] * high_tiles[0].pixel_size_um / 2
patch_x0, patch_x1 = high_xs.min() - patch_half_um, high_xs.max() + patch_half_um
patch_y0, patch_y1 = high_ys.min() - patch_half_um, high_ys.max() + patch_half_um
zoom_x0, zoom_x1 = patch_x0 - ZOOM_MARGIN_UM, patch_x1 + ZOOM_MARGIN_UM
zoom_y0, zoom_y1 = patch_y0 - ZOOM_MARGIN_UM, patch_y1 + ZOOM_MARGIN_UM

def _tile_overlaps_zoom(t):
    half = (t.image.shape[1] * t.pixel_size_um / 2, t.image.shape[0] * t.pixel_size_um / 2)
    return not (t.x_um + half[0] < zoom_x0 or t.x_um - half[0] > zoom_x1
                or t.y_um + half[1] < zoom_y0 or t.y_um - half[1] > zoom_y1)

low_tiles_shifted = [dataclasses.replace(t, x_um=t.x_um + SHIFT_DX_UM, y_um=t.y_um + SHIFT_DY_UM)
                     for t in low_tiles]

low_near_zoom_before = [t for t in low_tiles if _tile_overlaps_zoom(t)]
low_near_zoom_after  = [t for t in low_tiles_shifted if _tile_overlaps_zoom(t)]
print(f"Low-mag tiles near the high-mag patch: {len(low_near_zoom_before)} (before), "
      f"{len(low_near_zoom_after)} (after)")

zoom_canvas_before = assemble_mosaic_canvas(low_near_zoom_before, working_pixel_um=ZOOM_PIXEL_UM)
zoom_canvas_after  = assemble_mosaic_canvas(low_near_zoom_after,  working_pixel_um=ZOOM_PIXEL_UM)

def _um_box_to_px(canvas, x0, y0, x1, y1):
    col0, row0 = (x0 - canvas.origin_um[0]) / canvas.pixel_size_um, (y0 - canvas.origin_um[1]) / canvas.pixel_size_um
    col1, row1 = (x1 - canvas.origin_um[0]) / canvas.pixel_size_um, (y1 - canvas.origin_um[1]) / canvas.pixel_size_um
    return col0, row0, col1 - col0, row1 - row0

fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
for ax, canvas, title in (
    (axes[0], zoom_canvas_before, "before shift"),
    (axes[1], zoom_canvas_after,  f"after shift (dX={SHIFT_DX_UM}, dY={SHIFT_DY_UM} um)"),
):
    covered_vals = canvas.image[canvas.covered]
    vmin, vmax = np.percentile(covered_vals, [1, 99]) if covered_vals.size else (0, 1)
    ax.imshow(canvas.image, cmap="gray", vmin=vmin, vmax=vmax)
    col, row, w, h = _um_box_to_px(canvas, patch_x0, patch_y0, patch_x1, patch_y1)
    ax.add_patch(mpatches.Rectangle((col, row), w, h, linewidth=2, linestyle="--",
                                     edgecolor="tab:red", facecolor="none"))
    ax.set_title(title); ax.axis("off")
fig.suptitle(f"{SAMPLE_NAME}: does the high-mag patch's real footprint (dashed box) sit on tissue?")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step2_footprint_on_tissue_before_after.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step2_footprint_on_tissue_before_after.png'}")
print("Visually confirm the RIGHT panel's dashed box sits on continuous gray tissue "
      "texture (not black background) before trusting anything below -- if the LEFT "
      "panel already does instead, or neither does, SHIFT_DX_UM/SHIFT_DY_UM (or their "
      "sign) need correcting.")

## 6 — Step 3: shifted mosaic canvas + current (already-imaging) positions overlay

Assembles the mosaic from the SHIFTED low-mag tiles only (same
`MOSAIC_KEEP_OBJECTIVES`/`WORKING_PIXEL_UM` convention as the local
`02_create_boundary_from_mosaic.ipynb`), then overlays the positions file
currently being imaged -- if the original boundary really was derived from
the unshifted mosaic, this should show the current FOV grid sitting
offset from the real tissue.

In [ ]:
shifted_canvas = assemble_mosaic_canvas(
    [t for t in low_tiles_shifted if t.objective_name in MOSAIC_KEEP_OBJECTIVES],
    working_pixel_um=WORKING_PIXEL_UM,
)

current_positions = np.loadtxt(CURRENT_POSITIONS_PATH, delimiter=",")
print(f"Current positions: {len(current_positions)} FOV(s) from {CURRENT_POSITIONS_PATH.name}")

def _um_to_px(canvas, x, y):
    return ((np.asarray(x) - canvas.origin_um[0]) / canvas.pixel_size_um,
             (np.asarray(y) - canvas.origin_um[1]) / canvas.pixel_size_um)

px, py = _um_to_px(shifted_canvas, current_positions[:, 0], current_positions[:, 1])

fig, ax = plt.subplots(figsize=(9, 9))
covered_vals = shifted_canvas.image[shifted_canvas.covered]
vmin, vmax = np.percentile(covered_vals, [1, 99]) if covered_vals.size else (0, 1)
ax.imshow(shifted_canvas.image, cmap="gray", vmin=vmin, vmax=vmax)
ax.scatter(px, py, s=3, c="tab:orange", alpha=0.6, label=f"current positions ({len(current_positions)})")
ax.set_title(f"{SAMPLE_NAME}: shifted (corrected) mosaic vs. currently-imaging FOV grid")
ax.legend(); ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step3_shifted_mosaic_vs_current_fovs.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step3_shifted_mosaic_vs_current_fovs.png'}")

## 7 — Step 4: re-segment the shifted mosaic, rebuild the FOV grid

Same `segment_mosaic_tissue` parameters as the local
`02_create_boundary_from_mosaic.ipynb`, and the same `build_boundary_path`
call (`direction="vertical"`, `return_side="top"`) the local
`02_create_positions_from_boundaries.ipynb` uses for this sample's single-
tissue/single-boundary ("legacy") layout -- holes are loaded from the
EXISTING `positions/boundaries/from_mosaic/hole*.txt` files (holes don't
move; only the outer tissue boundary was ever derived from the un-shifted
mosaic's own threshold trace) and shifted the same way.

In [ ]:
BOUNDARY_DIR = POSITIONS_DIR / "boundaries" / "from_mosaic"

holes_raw = load_hole_polygons(BOUNDARY_DIR)
from shapely.affinity import translate
holes_shifted = [translate(h, xoff=SHIFT_DX_UM, yoff=SHIFT_DY_UM) for h in holes_raw]
print(f"Loaded + shifted {len(holes_shifted)} hole polygon(s)")

segmentation = segment_mosaic_tissue(
    shifted_canvas,
    threshold           = THRESHOLD,
    smooth_sigma_um     = SMOOTH_SIGMA_UM,
    close_radius_um     = CLOSE_RADIUS_UM,
    open_radius_um      = OPEN_RADIUS_UM,
    margin_um           = MARGIN_UM,
    min_tissue_area_um2 = MIN_TISSUE_AREA_UM2,
    min_hole_area_um2   = MIN_HOLE_AREA_UM2,
    min_island_area_um2 = MIN_ISLAND_AREA_UM2,
    simplify_tol_um     = SIMPLIFY_TOL_UM,
)
print(f"Threshold used : {segmentation.threshold:.1f}")
print(f"Tissue pieces  : {len(segmentation.tissue_polygons)}")
print(f"Holes (from mosaic threshold, not used below -- see markdown): {len(segmentation.hole_polygons)}")

if len(segmentation.tissue_polygons) != 1:
    raise ValueError(
        f"Expected exactly 1 tissue piece for this sample's known single-boundary "
        f"layout, got {len(segmentation.tissue_polygons)} -- inspect the plot below "
        f"and adjust THRESHOLD/morphology parameters before continuing."
    )
new_tissue_polygon = segmentation.tissue_polygons[0]

ax = plot_mosaic_segmentation(shifted_canvas, segmentation)
ax.figure.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step4_new_segmentation.png", dpi=150)
plt.show()

new_path = build_boundary_path(
    new_tissue_polygon, holes_shifted, step_size_um, fov_size_um,
    direction=SCAN_DIRECTION, return_side="top",
)
print(f"New FOV grid: {len(new_path)} FOV(s)")

## 8 — Step 5: overlay OLD vs. NEW positions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(current_positions[:, 0], current_positions[:, 1],
           s=6, c="tab:orange", alpha=0.6, label=f"OLD (currently imaging, {len(current_positions)})")
ax.scatter(new_path[:, 0], new_path[:, 1],
           s=6, c="tab:green", alpha=0.6, label=f"NEW (corrected, {len(new_path)})")
ax.set_title(f"{SAMPLE_NAME}: OLD vs. NEW FOV positions")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step5_old_vs_new_positions.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step5_old_vs_new_positions.png'}")

## 9 — Step 6: classify NEW FOVs as already-covered vs. MISSING

For each NEW FOV, find its nearest OLD FOV (by center distance) and compute
the overlap area between their two `fov_size_um` x `fov_size_um` axis-
aligned tiles. A NEW FOV counts as already covered if that overlap exceeds
`OVERLAP_THRESHOLD` (default 50%) of one tile's area; otherwise it's
MISSING -- real tissue area the old (mis-positioned) grid never imaged.

In [ ]:
def tile_overlap_fraction(center_a, center_b, tile_size):
    dx = abs(center_a[0] - center_b[0])
    dy = abs(center_a[1] - center_b[1])
    overlap_x = max(0.0, tile_size - dx)
    overlap_y = max(0.0, tile_size - dy)
    return (overlap_x * overlap_y) / (tile_size * tile_size)


old_tree = cKDTree(current_positions)
nearest_dist, nearest_idx = old_tree.query(new_path, k=1)

overlap_fracs = np.array([
    tile_overlap_fraction(new_path[i], current_positions[nearest_idx[i]], fov_size_um)
    for i in range(len(new_path))
])
missing_mask = overlap_fracs <= OVERLAP_THRESHOLD
missing_coords_unordered = new_path[missing_mask]   # NEW's own scan order -- see section 10 for why this isn't used directly

print(f"NEW FOVs                 : {len(new_path)}")
print(f"Already covered (>{OVERLAP_THRESHOLD:.0%} overlap with nearest OLD FOV): {(~missing_mask).sum()}")
print(f"MISSING (<= {OVERLAP_THRESHOLD:.0%} overlap) : {missing_mask.sum()}")

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(current_positions[:, 0], current_positions[:, 1],
           s=6, c="0.7", label=f"OLD ({len(current_positions)})")
ax.scatter(new_path[~missing_mask, 0], new_path[~missing_mask, 1],
           s=6, c="tab:green", label=f"NEW, already covered ({(~missing_mask).sum()})")
ax.scatter(missing_coords_unordered[:, 0], missing_coords_unordered[:, 1],
           s=10, c="tab:red", label=f"NEW, MISSING ({missing_mask.sum()})")
ax.set_title(f"{SAMPLE_NAME}: missing FOVs (never imaged by the old grid)")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step6_missing_fovs.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step6_missing_fovs.png'}")

## 10 — Step 7: re-order the MISSING FOVs into their own short-travel loop

Simply keeping `new_path`'s own boustrophedon order for JUST the missing
subset does NOT give a sensible loop: `new_path` snakes through the WHOLE
tissue column by column, and any single column can contribute anywhere
from zero to all of its FOVs to the missing set depending on where that
column's own boundary happens to differ from the old grid -- extracting a
boolean-masked subsequence from that full snake jumps unpredictably
between whichever fragments happen to be missing in each column, in
column order, not in a locally-short path.

Tried a simple per-column boustrophedon re-sort first (group by real
lattice column, sort each column by the other axis, alternate direction
column to column -- the same convention `generate_scanning_path` uses for
a full grid). Measuring actual total travel (`get_path_stats`) showed this
was NOT reliably better than the naive subsequence order: a single lattice
column can itself contain more than one disconnected run of missing FOVs
(e.g. two separate notches crossing the same column), and sorting that
column's points by one axis alone still jumps across the gap between runs.
**Used a greedy nearest-neighbor walk instead** -- starting from the
missing FOV closest to `current_positions`'s own last point (so the
transit from the existing loop's end into this new loop is also short, not
just the loop's own internal travel), then repeatedly stepping to the
nearest not-yet-visited missing FOV. Simple and never takes a large hop
when a smaller one is available; not a guaranteed-optimal tour (true
optimal touring is NP-hard), but directly measured below to confirm it
beats both alternatives on this real data before using it.

In [ ]:
def nearest_neighbor_order(coords, start_point):
    remaining = coords.copy()
    order = []
    current = np.asarray(start_point, dtype=float)
    while len(remaining):
        dists = np.linalg.norm(remaining - current, axis=1)
        idx = int(np.argmin(dists))
        current = remaining[idx]
        order.append(current)
        remaining = np.delete(remaining, idx, axis=0)
    return np.array(order)


def boustrophedon_order(coords, step_size_um, direction="vertical"):
    primary_axis, secondary_axis = (0, 1) if direction == "vertical" else (1, 0)
    col_idx = np.round((coords[:, primary_axis] - coords[:, primary_axis].min()) / step_size_um).astype(int)
    chunks = []
    for col in sorted(set(col_idx)):
        members = coords[col_idx == col]
        members = members[np.argsort(members[:, secondary_axis])]
        if col % 2 == 1:
            members = members[::-1]
        chunks.append(members)
    return np.concatenate(chunks, axis=0)


missing_coords_column_sorted = boustrophedon_order(missing_coords_unordered, step_size_um, direction=SCAN_DIRECTION)
missing_coords_nearest_neighbor = nearest_neighbor_order(missing_coords_unordered, current_positions[-1])

candidates = {
    "new_path's own subsequence order": missing_coords_unordered,
    "per-column boustrophedon re-sort": missing_coords_column_sorted,
    "greedy nearest-neighbor walk":     missing_coords_nearest_neighbor,
}
for label, coords in candidates.items():
    length, max_step = get_path_stats(coords)
    print(f"{label:38s}: {length/1000:6.2f} mm total (max single step {max_step:6.0f} um)")

missing_coords = missing_coords_nearest_neighbor
print(f"\nUsing: greedy nearest-neighbor walk")

## 11 — Step 8-9: append the re-ordered MISSING FOVs, save a new positions file

Appended directly after the current (OLD) positions, so the existing
imaging loop's own order is completely undisturbed; only a new loop over
the appended tail needs to be added in Dave.

Writes a NEW file (`_added` suffix) -- never overwrites the original
positions file this sample is currently being imaged from.

In [ ]:
added_positions = np.concatenate([current_positions, missing_coords], axis=0)
save_positions_array(added_positions, ADDED_POSITIONS_PATH)

print(f"Wrote {len(added_positions)} FOV(s) ({len(current_positions)} original + "
      f"{len(missing_coords)} appended missing) to:")
print(f"  {ADDED_POSITIONS_PATH}")

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(current_positions[:, 0], current_positions[:, 1],
           s=6, c="tab:orange", label=f"original ({len(current_positions)})")
ax.scatter(missing_coords[:, 0], missing_coords[:, 1],
           s=10, c="tab:red", label=f"appended, missing ({len(missing_coords)})")
ax.plot(missing_coords[:, 0], missing_coords[:, 1], "-", lw=0.5, c="tab:red", alpha=0.5)
ax.set_title(f"{SAMPLE_NAME}: final positions file ({ADDED_POSITIONS_PATH.name})")
ax.legend(); ax.axis("equal"); ax.invert_yaxis()
ax.set_xlabel("stage x (um)"); ax.set_ylabel("stage y (um)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / f"{NOTEBOOK_NAME}.step7_final_added_positions.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{NOTEBOOK_NAME}.step7_final_added_positions.png'}")
print()
print("Review every figure in", FIGURES_DIR, "before using this file -- in particular "
      "step2 (does the shift actually align the two objectives?) and step6/7 (do the "
      "MISSING FOVs look like a real tissue-edge strip, not noise).")